In [19]:
import pandas as pd

## Implementing the retrieval metrics


In [33]:
def precision(k:int, q_rels:list, q_ret:list) -> float:
    cpt = 0
    for doc in q_ret[:k]:
        if doc in q_rels:
            cpt += 1
    return cpt / k

def recall(k:int, q_rels:list, q_ret:list) -> float:
    cpt = 0
    for doc in q_ret[:k]:
        if doc in q_rels:
            cpt += 1
    return cpt / len(q_rels) if q_rels else 0

## Connect to the VDB


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "hybrid_collection"

def retrieve_relevant_chunks(query: str, k: int, retrieval_mode: RetrievalMode = RetrievalMode.HYBRID) -> list:
    # Import embedding model
    dense_embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25") # Using the qdrant bm25 model for sparse embeddings



    vector_store = QdrantVectorStore.from_existing_collection(
        embedding=dense_embeddings, #embeddings uses the dense embedding (classical embedding that we know)
        sparse_embedding=sparse_embeddings,  #sparse embeddings mean the bm25
        retrieval_mode=RetrievalMode.HYBRID, # Select the retrieval mode : DENSE / HYBRID / SPARSE
        url=QDRANT_URL,
        collection_name=COLLECTION_NAME,
    )


    relevant_chunks = vector_store.client.query_points(
        collection_name=COLLECTION_NAME,
        retrieval_mode=retrieval_mode,
        query=vector_store.embeddings.embed_query(query),
        with_payload=True,
        limit=k,
        
    )
    chunks_ids = []
    for chunk in relevant_chunks.points:
        chunks_ids.append(chunk.id)
        
    return chunks_ids

In [29]:
import json


benchmark_questions = pd.read_json("../questions.jsonl", lines=True)
benchmark_questions.index = benchmark_questions.query_id


with open("../qrels.json", "r", encoding="utf-8") as file:
    qrels = json.load(file)

In [38]:
K = 10
q_id = "Q001"

query = benchmark_questions.loc[q_id].question
q_rels = qrels[q_id]

q_ret = retrieve_relevant_chunks(query, K)

print(f' precision : {precision(K, q_rels, q_ret)}, recall :  {recall(K, q_rels, q_ret)}')

 precision : 0.2, recall :  1.0
